In [19]:
import pandas as pd

df = pd.read_csv('loan_risk_prediction_dataset.csv')

In [20]:
df.head()

,Age,Income,LoanAmount,CreditScore,YearsExperience,Gender,Education,City,EmploymentType,LoanApproved
0,56,48353.0,31258.0,675.0,20,Female,High School,Houston,Unemployed,0
1,69,57462.0,23262.0,586.0,6,Male,High School,San Francisco,Self-Employed,0
2,46,44219.0,26530.0,781.0,26,Male,PhD,Houston,Self-Employed,1
3,32,56307.0,11531.0,549.0,11,Male,NaN,New York,Unemployed,0
4,60,37034.0,27871.0,500.0,19,Female,High School,Chicago,Unemployed,0


In [21]:
df.shape

(5000, 10)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Age              5000 non-null   int64  
 1   Income           4804 non-null   float64
 2   LoanAmount       5000 non-null   float64
 3   CreditScore      4806 non-null   float64
 4   YearsExperience  5000 non-null   int64  
 5   Gender           5000 non-null   object 
 6   Education        4802 non-null   object 
 7   City             5000 non-null   object 
 8   EmploymentType   5000 non-null   object 
 9   LoanApproved     5000 non-null   int64  
dtypes: float64(3), int64(3), object(4)
memory usage: 390.8+ KB


In [23]:
df.describe()

,Age,Income,LoanAmount,CreditScore,YearsExperience,LoanApproved
count,5000.000000,4804.000000,5000.000000,4806.000000,5000.000000,5000.000000
mean,43.584600,49738.123022,19870.768600,575.494590,19.599000,0.230200
std,14.919094,15101.361851,8046.542413,160.550839,11.516837,0.421003
min,18.000000,-3731.000000,-10059.000000,300.000000,0.000000,0.000000
25%,31.000000,39608.500000,14455.250000,433.000000,10.000000,0.000000
50%,43.000000,49488.000000,19842.500000,579.000000,20.000000,0.000000
75%,56.000000,59917.000000,25326.750000,712.000000,29.000000,0.000000
max,69.000000,99146.000000,48353.000000,849.000000,39.000000,1.000000


In [24]:
df['Income'] = df['Income'].fillna(df['Income'].median())
df['CreditScore'] = df['CreditScore'].fillna(df['CreditScore'].median())
df['Education'] = df['Education'].fillna(df['Education'].mode()[0])


In [25]:
df = df[(df['Income'] >= 0) & (df['LoanAmount'] >= 0)]

In [26]:
df.shape

(4969, 10)

In [27]:
df.isnull().sum()

Age                0
Income             0
LoanAmount         0
CreditScore        0
YearsExperience    0
Gender             0
Education          0
City               0
EmploymentType     0
LoanApproved       0
dtype: int64

In [28]:
df.duplicated().sum()

np.int64(0)

In [30]:
df['Gender'].value_counts()

Gender
Male      2522
Female    2447
Name: count, dtype: int64

In [31]:
df['City'].value_counts()

City
Chicago          1288
San Francisco    1254
Houston          1226
New York         1201
Name: count, dtype: int64

In [32]:
df['EmploymentType'].value_counts()

EmploymentType
Self-Employed    1720
Unemployed       1647
Salaried         1602
Name: count, dtype: int64

In [33]:
df['LoanApproved'].value_counts()
df['LoanApproved'].value_counts(normalize=True)

LoanApproved
0    0.769169
1    0.230831
Name: proportion, dtype: float64

In [34]:
df.groupby('LoanApproved')[['Age', 'Income', 'LoanAmount', 'CreditScore', 'YearsExperience']].mean()

,Age,Income,LoanAmount,CreditScore,YearsExperience
LoanApproved,,,,,
0,43.633961,48244.911826,19937.369440,535.492151,19.620356
1,43.360070,54779.014821,20219.344377,708.229294,19.491718


In [36]:
df.groupby('Gender')['LoanApproved'].mean()

Gender
Female    0.223539
Male      0.237906
Name: LoanApproved, dtype: float64

In [37]:
df.groupby('Education')['LoanApproved'].mean()

Education
Bachelors      0.219616
High School    0.221750
Masters        0.232831
PhD            0.251050
Name: LoanApproved, dtype: float64

In [38]:
df.groupby('City')['LoanApproved'].mean()

City
Chicago          0.235248
Houston          0.218597
New York         0.230641
San Francisco    0.238437
Name: LoanApproved, dtype: float64

In [39]:
df.groupby('EmploymentType')['LoanApproved'].mean()

EmploymentType
Salaried         0.333333
Self-Employed    0.326163
Unemployed       0.031573
Name: LoanApproved, dtype: float64

In [40]:
df.corr(numeric_only=True)['LoanApproved'].sort_values(ascending=False)

LoanApproved       1.000000
CreditScore        0.462452
Income             0.187006
LoanAmount         0.015071
YearsExperience   -0.004705
Age               -0.007734
Name: LoanApproved, dtype: float64

In [44]:
df['LoanToIncomeRatio'] = df['LoanAmount'] / df['Income']
df.groupby('LoanApproved')['LoanToIncomeRatio'].mean()

LoanApproved
0    0.526634
1    0.388294
Name: LoanToIncomeRatio, dtype: float64

In [45]:
df.groupby('EmploymentType')['CreditScore'].mean()

EmploymentType
Salaried         576.239076
Self-Employed    574.797674
Unemployed       575.108075
Name: CreditScore, dtype: float64

In [46]:
from sqlalchemy import create_engine

In [47]:
from urllib.parse import quote_plus

username = 'postgres'
password = quote_plus('Sneha@0033') 
host = 'localhost'
port = '5432'
database = 'loan_risk_db'

engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database}')

In [48]:
df.to_sql('loans', engine, if_exists='replace', index=False)

969

In [49]:
import pandas as pd
pd.read_sql('SELECT * FROM loans LIMIT 5;', engine)

,Age,Income,LoanAmount,CreditScore,YearsExperience,Gender,Education,City,EmploymentType,LoanApproved,LoanToIncomeRatio
0,56,48353.0,31258.0,675.0,20,Female,High School,Houston,Unemployed,0,0.646454
1,69,57462.0,23262.0,586.0,6,Male,High School,San Francisco,Self-Employed,0,0.404824
2,46,44219.0,26530.0,781.0,26,Male,PhD,Houston,Self-Employed,1,0.599968
3,32,56307.0,11531.0,549.0,11,Male,Bachelors,New York,Unemployed,0,0.204788
4,60,37034.0,27871.0,500.0,19,Female,High School,Chicago,Unemployed,0,0.752579
